In [2]:
import numpy as np
import pandas as pd
import os
import sys
import zipfile
import subprocess

from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from tqdm.notebook import tqdm
from copy import deepcopy

import json

In [3]:
DATASET = 'ml-1m' 
RAW_PATH = os.path.join('./', DATASET)

RANDOM_SEED = 0
NEG_ITEMS = 99

# Load data

1. Load interaction data and item metadata
2. Filter out items with less than 5 interactions
3. Calculate basic statistics

In [3]:
import requests

# download data if not exists

url = f'http://files.grouplens.org/datasets/movielens/{DATASET}.zip'
zip_path = os.path.join(RAW_PATH, DATASET + '.zip')

os.makedirs(RAW_PATH, exist_ok=True)
if not os.path.exists(zip_path):
    # 1. 下载
    print('Downloading', url)
    r = requests.get(url, stream=True, timeout=30)
    r.raise_for_status()
    with open(zip_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1<<20):
            if chunk:
                f.write(chunk)

    # 2. 解压
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_PATH)
else:
    print('Data already exists.')
print('Done.')

Data already exists.
Done.


In [5]:
with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_PATH)

In [4]:
# read interaction data
interactions = []
user_freq, item_freq = dict(), dict()
file = os.path.join(RAW_PATH,DATASET,"ratings.dat")
with open(file) as F:
    header = 0
    for line in tqdm(F):
        if header == 1:
            header = 0
            continue
        line = line.strip().split("::")
        uid, iid, rating, time = line[0], line[1], float(line[2]), float(line[3])
        if rating >= 4:
            label = 1
        else:
            label = 0
        interactions.append([uid,time,iid,label])
        if int(label)==1:
            user_freq[uid] = user_freq.get(uid,0)+1
            item_freq[iid] = item_freq.get(iid,0)+1

0it [00:00, ?it/s]

In [5]:
# 5-core filtering
select_uid, select_iid = [],[]
while len(select_uid)<len(user_freq) or len(select_iid)<len(item_freq):
    select_uid, select_iid = [],[]
    for u in user_freq:
        if user_freq[u]>=5:
            select_uid.append(u)
    for i in item_freq:
        if item_freq[i]>=5:
            select_iid.append(i)
    print("User: %d/%d, Item: %d/%d"%(len(select_uid),len(user_freq),len(select_iid),len(item_freq)))

    select_uid = set(select_uid)
    select_iid = set(select_iid)
    user_freq, item_freq = dict(), dict()
    interactions_5core = []
    for line in tqdm(interactions):
        uid, iid, label = line[0], line[2], line[-1]
        if uid in select_uid and iid in select_iid:
            interactions_5core.append(line)
            if int(label)==1:
                user_freq[uid] = user_freq.get(uid,0)+1
                item_freq[iid] = item_freq.get(iid,0)+1
    interactions = interactions_5core

User: 6034/6038, Item: 3125/3533


  0%|          | 0/1000209 [00:00<?, ?it/s]

In [6]:
print("Selected Interactions: %d, Users: %d, Items: %d"%(len(interactions),len(select_uid),len(select_iid)))

Selected Interactions: 994338, Users: 6034, Items: 3125


In [7]:
# Get timestamp
ts = []
for i in tqdm(range(len(interactions))):
    ts.append(datetime.fromtimestamp(interactions[i][1]))

  0%|          | 0/994338 [00:00<?, ?it/s]

In [8]:
# Construct and Save 5 core results with situation context
interaction_df = pd.DataFrame(interactions,columns = ["user_id","time","news_id","label"])
interaction_df['timestamp'] = ts
interaction_df['hour'] = interaction_df['timestamp'].apply(lambda x: x.hour)
interaction_df['weekday'] = interaction_df['timestamp'].apply(lambda x: x.weekday())
interaction_df['date'] = interaction_df['timestamp'].apply(lambda x: x.date())
# ?这啥啊这是？
def get_time_range(hour): # according to the Britannica dictionary
    # https://www.britannica.com/dictionary/eb/qa/parts-of-the-day-early-morning-late-morning-etc
    if hour>=5 and hour<=8:
        return 0
    if hour>8 and hour<11:
        return 1
    if hour>=11 and hour<=12:
        return 2
    if hour>12 and hour<=15:
        return 3
    if hour>15 and hour<=17:
        return 4
    if hour>=18 and hour<=19:
        return 5
    if hour>19 and hour<=21:
        return 6
    if hour>21:
        return 7
    return 8 # 0-4 am

interaction_df['period'] = interaction_df.hour.apply(lambda x: get_time_range(x))
min_date = interaction_df.date.min()
interaction_df['day'] = (interaction_df.date - min_date).apply(lambda x: x.days)


interaction_df["user_id"] = interaction_df["user_id"].astype(int)
interaction_df["item_id"] = interaction_df["news_id"].astype(int)
interaction_df.to_csv("interaction_5core.csv",index=False)


---
# 为DiffRec项目准备数据集
仅需要保留 user_id, item_id, time 三列即可。

In [11]:

GDR_PATH='./ML_1MGDR/'
os.makedirs(GDR_PATH,exist_ok=True)

if os.path.exists('./interaction_5core.csv'):
    interaction_df = pd.read_csv('./interaction_5core.csv')
    print('✅ 读取 interaction 数据集成功')

✅ 读取 interaction 数据集成功


In [12]:
# ==========================================
# Leave-Last-Out
# ==========================================

GDR_PATH='./ML_1MGDR/'
os.makedirs(GDR_PATH, exist_ok=True)

# 排序
interaction_gdr = interaction_df[['user_id', 'item_id', 'time']].copy()
interaction_gdr = interaction_gdr.sort_values(['user_id', 'time'])

# 划分 Train/Dev/Test
train_data = []
dev_data = []
test_data = []

# 按用户分组处理
for user_id, group in tqdm(interaction_gdr.groupby('user_id')):
    # 如果交互记录太少（比如少于3条），全放进训练集，或者丢弃
    if len(group) < 3:
        train_data.append(group)
        continue

    items = group['item_id'].tolist()
    times = group['time'].tolist()

    test_data.append([user_id, items[-1], times[-1]])
    dev_data.append([user_id, items[-2], times[-2]])
    train_df_user = pd.DataFrame({
        'user_id': [user_id] * (len(items) - 2),
        'item_id': items[:-2],
        'time': times[:-2]
    })
    train_data.append(train_df_user)

train_df = pd.concat(train_data, ignore_index=True)
dev_df = pd.DataFrame(dev_data, columns=['user_id', 'item_id', 'time'])
test_df = pd.DataFrame(test_data, columns=['user_id', 'item_id', 'time'])

print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

train_df.to_csv(os.path.join(GDR_PATH, 'train.csv'), index=False, sep='\t')
dev_df.to_csv(os.path.join(GDR_PATH, 'dev.csv'), index=False, sep='\t')
test_df.to_csv(os.path.join(GDR_PATH, 'test.csv'), index=False, sep='\t')

print("数据处理完成")

  0%|          | 0/6034 [00:00<?, ?it/s]

Train size: 982270
Dev size:   6034
Test size:  6034
✅ 数据处理完成！Train/Dev/Test 包含相同的用户集合。


# Prepare data for Top-k Recommendation Task
1. Rename all interaction features
2. Split dataset into training, validation, and test
3. Re-assign IDs to user, item, and context; Save interaction files
4. Organize item metadata

In [9]:
TOPK_PATH='./ML_1MTOPK/'
os.makedirs(TOPK_PATH,exist_ok=True)


In [10]:
# copy & rename columns
interaction_pos = interaction_df.loc[interaction_df.label==1].copy() # retain positive interactions
interaction_pos.rename(columns={'hour':'c_hour_c','weekday':'c_weekday_c','period':'c_period_c','day':'c_day_f',
                              'user_id':'original_user_id'}, inplace=True)

In [11]:
# split training, validation, and test sets.
split_time1 = int(interaction_pos.c_day_f.max() * 0.8)
train = interaction_pos.loc[interaction_pos.c_day_f<=split_time1].copy()
val_test = interaction_pos.loc[(interaction_pos.c_day_f>split_time1)].copy()
val_test.sort_values(by='time',inplace=True)
split_time2 = int(interaction_pos.c_day_f.max() * 0.9)
val = val_test.loc[val_test.c_day_f<=split_time2].copy()
test = val_test.loc[val_test.c_day_f>split_time2].copy()

# Delete user&item in validation&test sets that not exist in training set
train_u, train_i = set(train.original_user_id.unique()), set(train.news_id.unique())
val_sel = val.loc[(val.original_user_id.isin(train_u))&(val.news_id.isin(train_i))].copy()
test_sel = test.loc[(test.original_user_id.isin(train_u))&(test.news_id.isin(train_i))].copy()
print("Train user: %d, item: %d"%(len(train_u),len(train_i)))
print("Validation user: %d, item:%d"%(val_sel.original_user_id.nunique(),val_sel.news_id.nunique()))
print("Test user: %d, item:%d"%(test_sel.original_user_id.nunique(),test_sel.news_id.nunique()))
train.label.sum(),train.label.mean(),val_sel.label.sum(),val_sel.label.mean(),test_sel.label.sum(),test_sel.label.mean()

Train user: 6032, item: 3125
Validation user: 214, item:1198
Test user: 226, item:1260


(568761, 1.0, 2562, 1.0, 2874, 1.0)

In [12]:
# Assign ids for users and items (to generate continous ids)
all_df = pd.concat([train,val_sel,test_sel],axis=0)
user2newid_topk = dict(zip(sorted(all_df.original_user_id.unique()),
                      range(1,all_df.original_user_id.nunique()+1)))

for df in [train,val_sel,test_sel]:
    df['user_id'] = df.original_user_id.apply(lambda x: user2newid_topk[x])

item2newid_topk = dict(zip(sorted(all_df.news_id.unique()),
                      range(1,all_df.news_id.nunique()+1)))
for df in [train,val_sel,test_sel]:
    df['item_id'] = df['news_id'].apply(lambda x: item2newid_topk[x])

all_df['user_id'] = all_df.original_user_id.apply(lambda x: user2newid_topk[x])
all_df['item_id'] = all_df['news_id'].apply(lambda x: item2newid_topk[x])

In [13]:
nu2nid = dict()
ni2nid = dict()
for i in user2newid_topk.keys():
    oi = int(i)
    nu2nid[oi] = user2newid_topk[i]

for i in item2newid_topk.keys():
    oi = int(i)
    ni2nid[oi] = item2newid_topk[i]
json.dump(nu2nid,open(os.path.join(TOPK_PATH,"user2newid.json"),'w'))
json.dump(ni2nid,open(os.path.join(TOPK_PATH,"item2newid.json"),'w'))

In [14]:
# generate negative items
def generate_negative(data_df,all_items,clicked_item_set,random_seed,neg_item_num=99):
    np.random.seed(random_seed)
    neg_items = np.random.choice(all_items, (len(data_df),neg_item_num))
    for i, uid in tqdm(enumerate(data_df['user_id'].values)):
        user_clicked = clicked_item_set[uid]
        for j in range(len(neg_items[i])):
            while neg_items[i][j] in user_clicked|set(neg_items[i][:j]):
                neg_items[i][j] = np.random.choice(all_items, 1)
    return neg_items.tolist()

clicked_item_set = dict()
for user_id, seq_df in all_df.groupby('user_id'):
    clicked_item_set[user_id] = set(seq_df['item_id'].values.tolist())
all_items = all_df.item_id.unique()
val_sel['neg_items'] = generate_negative(val_sel,all_items,clicked_item_set,random_seed=1)
test_sel['neg_items'] = generate_negative(test_sel,all_items,clicked_item_set,random_seed=2)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [15]:
select_columns = ['user_id','item_id','time','c_hour_c','c_weekday_c','c_period_c','c_day_f']
# select_columns = ['user_id','item_id','time']
train[select_columns].to_csv(os.path.join(TOPK_PATH,'train.csv'),sep="\t",index=False)
val_sel[select_columns+['neg_items']].to_csv(os.path.join(TOPK_PATH,'dev.csv'),sep="\t",index=False)
test_sel[select_columns+['neg_items']].to_csv(os.path.join(TOPK_PATH,'test.csv'),sep="\t",index=False)

In [16]:
# organize & save item metadata
item_meta = pd.read_csv(os.path.join(DATASET, "ml-1m/movies.dat"),
            sep='::',names=['movieId','title','genres'],encoding='latin-1',engine='python') # columns: movieId,title,genres
item_select = item_meta.loc[item_meta.movieId.isin(interaction_pos.news_id.unique())].copy()
item_select['item_id'] = item_select.movieId.apply(lambda x: item2newid_topk[x])
genres2id = dict(zip(sorted(item_select.genres.unique()),range(1,item_select.genres.nunique()+1)))
item_select['i_genre_c'] = item_select['genres'].apply(lambda x: genres2id[x])
title2id = dict(zip(sorted(item_select.title.unique()),range(1,item_select.title.nunique()+1)))
item_select['i_title_c'] = item_select['title'].apply(lambda x: title2id[x])

item_select[['item_id','i_genre_c','i_title_c']].to_csv(
    os.path.join(TOPK_PATH,'item_meta.csv'),sep="\t",index=False)